# Rooftop Solar Across Data Environments — run notebook

Runs the Austin (LiDAR) and Kathmandu (open DSM) pipelines and builds the
comparison figures. Read the README first. Replace placeholder GHI values with
cited Global Solar Atlas figures before reporting numbers.

In [ ]:
import sys, os
sys.path.insert(0, 'src')
import geopandas as gpd
import matplotlib.pyplot as plt
from solar_suitability import building_suitability

## 1. Austin — build DSM/DTM from the LiDAR tile
Run once from a terminal (PDAL), then the surfaces are on disk:
```
python src/point_cloud_to_surfaces.py --laz data/austin/tile.laz \
    --out-dir outputs/austin --resolution 1.0 --epsg 6578
```
Confirm `--epsg` against the CRS the script prints.

In [ ]:
# Austin: aggregate to OSM footprints. Fetch footprints for the tile's extent.
import osmnx as ox
import rasterio
from shapely.geometry import box

with rasterio.open('outputs/austin/dsm.tif') as src:
    b = src.bounds; crs = src.crs
aoi = gpd.GeoDataFrame(geometry=[box(*b)], crs=crs).to_crs(4326)
w,s,e,n = aoi.total_bounds
fp = ox.features_from_bbox(n, s, e, w, {'building': True})
fp = fp[fp.geometry.type.isin(['Polygon','MultiPolygon'])].to_crs(crs)
fp[['geometry']].to_file('outputs/austin/osm_footprints.geojson', driver='GeoJSON')
len(fp)

In [ ]:
austin = building_suitability(
    'outputs/austin/osm_footprints.geojson',
    'outputs/austin/dsm.tif',
    'outputs/austin/buildings_solar.geojson',
    annual_ghi_kwh_m2=1700.0,   # replace with cited Austin GHI
    max_slope_deg=35.0,
)
austin[['footprint_m2','usable_m2','mean_slope_deg','aspect_score','annual_kwh']].describe()

## 2. Kathmandu — transfer with open DSM
```
python src/kathmandu_transfer.py --dsm data/kathmandu/dsm.tif \
    --out outputs/kathmandu/buildings_solar.geojson --bbox 27.69 85.30 27.73 85.34
```

In [ ]:
ktm = gpd.read_file('outputs/kathmandu/buildings_solar.geojson')
ktm[['footprint_m2','usable_m2','mean_slope_deg','aspect_score','annual_kwh']].describe()

## 3. Comparison figure
The point of the project: same method, two data environments.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
austin.plot(column='annual_kwh', legend=True, ax=axes[0], cmap='viridis')
axes[0].set_title('Austin (1 m LiDAR DSM)\nper-roof suitability')
axes[0].axis('off')
ktm.plot(column='annual_kwh', legend=True, ax=axes[1], cmap='viridis')
axes[1].set_title('Kathmandu (30 m open DSM)\nfootprint-level screen')
axes[1].axis('off')
plt.tight_layout()
os.makedirs('docs', exist_ok=True)
plt.savefig('docs/comparison.png', dpi=150, bbox_inches='tight')
plt.show()